# 🇹🇷 Qwen3-TTS Türkçe Fine-Tuning — v16

Tüm veriler ve çıktılar **Google Drive**'a kaydedilir.

**Akış:**
1. GPU Kontrolü
2. Kurulum
3. Drive Bağlantısı & Yol Ayarları
4. Veri Hazırlama (CSV → JSONL)
5. Audio Code Çıkarma
6. Veri Temizleme
7. Referans Ses 24kHz Dönüşümü
8. Eğitim Ayarları
9. Script Yazma (dataset.py + sft_v16.py)
10. Eğitim
11. Inference Testi
12. Bellek Temizliği

> ⚠️ **A100 GPU gereklidir.** Runtime → Change runtime type → A100 GPU

---
## 1️⃣ GPU Kontrolü

In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), "GPU bulunamadı! Runtime > Change runtime type > GPU seç."

name = torch.cuda.get_device_name(0)
mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU  : {name}")
print(f"VRAM : {mem:.1f} GB")

if "A100" in name:
    print("✓ A100 — optimum ayarlar aktif")
elif mem >= 16:
    print("⚠ A100 değil ama yeterli VRAM — batch_size düşürülmeli")
else:
    raise RuntimeError("T4 Free ile bu notebook çalışmaz. En az V100 gerekli.")

---
## 2️⃣ Kurulum

> Her Colab oturumunda çalıştır.

In [ ]:
import os, sys

!apt-get install -y sox libsox-dev libsox-fmt-all > /dev/null 2>&1
!pip install -q accelerate transformers safetensors datasets pandas \
             torchaudio librosa soundfile tensorboard qwen-tts

if not os.path.exists('/content/Qwen3-TTS'):
    !git clone https://github.com/QwenLM/Qwen3-TTS.git /content/Qwen3-TTS
    print("Repo klonlandı")
else:
    print("Repo zaten mevcut")

%cd /content/Qwen3-TTS
!pip install -q -e .

for p in ['/content/Qwen3-TTS', '/content/Qwen3-TTS/finetuning']:
    if p not in sys.path:
        sys.path.insert(0, p)

print("✓ Kurulum tamamlandı.")

---
## 3️⃣ Drive Bağlantısı & Yol Ayarları

Aşağıdaki değişkenleri kendi Drive yapınıza göre ayarlayın.

In [ ]:
import os, json
from google.colab import drive

drive.mount('/content/drive')

# ============================================================
# YOLLAR — Kendi Drive yapınıza göre ayarlayın
# ============================================================
DATASET_DIR    = "/content/drive/MyDrive/Turkish_TTS_Dataset"
WAVS_DIR       = os.path.join(DATASET_DIR, "wavs")
REF_AUDIO_PATH = os.path.join(DATASET_DIR, "ceren.wav")
REF_TEXT       = "Hayvanlar için hayatlarını tehlikeye atmaya hazır insanlar var."
OUTPUT_BASE    = "/content/drive/MyDrive/Qwen3_TTS_Output"
# ============================================================

os.makedirs(OUTPUT_BASE, exist_ok=True)

assert os.path.isdir(DATASET_DIR), f"Veri seti klasörü bulunamadı: {DATASET_DIR}"
assert os.path.isdir(WAVS_DIR),    f"wavs/ klasörü bulunamadı: {WAVS_DIR}"
assert os.path.isfile(os.path.join(DATASET_DIR, "metadata_cleaned.csv")), "metadata_cleaned.csv bulunamadı!"

wav_count = len([f for f in os.listdir(WAVS_DIR) if f.endswith(".wav")])
print(f"📁 Veri seti    : {DATASET_DIR}")
print(f"🎵 WAV sayısı   : {wav_count:,}")
print(f"💾 Çıktı klasörü: {OUTPUT_BASE}")

if os.path.isfile(REF_AUDIO_PATH):
    print(f"🎤 Referans ses : {REF_AUDIO_PATH} ✓")
else:
    print(f"⚠  Referans ses bulunamadı: {REF_AUDIO_PATH}")

---
## 4️⃣ Veri Hazırlama (CSV → JSONL)

`metadata_cleaned.csv` → `train_raw.jsonl` dönüşümü.
Daha önce oluşturulduysa bu hücre atlanabilir.

In [ ]:
import json, random, os

RAW_JSONL   = os.path.join(OUTPUT_BASE, "train_raw.jsonl")
MAX_SAMPLES = 20000  # Kullanılacak maksimum örnek sayısı

if os.path.isfile(RAW_JSONL):
    line_count = sum(1 for _ in open(RAW_JSONL))
    print(f"⏭ train_raw.jsonl zaten mevcut ({line_count:,} satır) — atlanıyor.")
    print("Yeniden oluşturmak için dosyayı Drive'dan silin.")
else:
    csv_path = os.path.join(DATASET_DIR, "metadata_cleaned.csv")
    valid_samples = []

    with open(csv_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split('|')
            if len(parts) < 3:
                continue
            file_id         = parts[0].strip()
            normalized_text = parts[2].strip()
            wav_path        = os.path.join(WAVS_DIR, f"{file_id}.wav")
            if not os.path.exists(wav_path):
                continue
            if not normalized_text or len(normalized_text) < 2:
                continue
            valid_samples.append({
                "audio":     wav_path,
                "text":      normalized_text,
                "ref_audio": wav_path   # Her ses kendi referansı
            })

    print(f"Toplam geçerli örnek: {len(valid_samples):,}")
    random.seed(42)
    random.shuffle(valid_samples)
    selected = valid_samples[:MAX_SAMPLES]

    with open(RAW_JSONL, 'w', encoding='utf-8') as f:
        for s in selected:
            f.write(json.dumps(s, ensure_ascii=False) + '\n')

    print(f"✅ Drive'a kaydedildi: {RAW_JSONL} ({len(selected):,} örnek)")

---
## 5️⃣ Audio Code Çıkarma

WAV dosyalarını codec tokenlarına dönüştürür.
Daha önce yapıldıysa bu hücre atlanabilir.

> ⏱ ~20k örnek için yaklaşık 15-20 dakika sürer.

In [ ]:
import os

WORK_DIR          = "/content/Qwen3-TTS/finetuning"
TOKENIZER_MODEL   = "Qwen/Qwen3-TTS-Tokenizer-12Hz"
RAW_JSONL         = os.path.join(OUTPUT_BASE, "train_raw.jsonl")
TRAIN_CODES_JSONL = os.path.join(OUTPUT_BASE, "train_with_codes.jsonl")

if os.path.isfile(TRAIN_CODES_JSONL):
    line_count = sum(1 for _ in open(TRAIN_CODES_JSONL))
    print(f"⏭ train_with_codes.jsonl zaten mevcut ({line_count:,} satır) — atlanıyor.")
else:
    !cd {WORK_DIR} && python prepare_data.py \
        --device cuda:0 \
        --tokenizer_model_path {TOKENIZER_MODEL} \
        --input_jsonl {RAW_JSONL} \
        --output_jsonl {TRAIN_CODES_JSONL}

    if os.path.isfile(TRAIN_CODES_JSONL):
        lc = sum(1 for _ in open(TRAIN_CODES_JSONL))
        print(f"\n✅ Drive'a kaydedildi: {TRAIN_CODES_JSONL} ({lc:,} örnek)")
    else:
        print("\n❌ HATA: train_with_codes.jsonl oluşturulamadı!")

---
## 6️⃣ Veri Temizleme

- Çok kısa/uzun metin filtresi
- Apostrof ve tırnak temizliği
- Codec uzunluk filtresi
- `ref_audio` → her sesin kendisi (eğitim kalitesi için kritik)

In [ ]:
import json, re, os

TRAIN_CODES_JSONL = os.path.join(OUTPUT_BASE, "train_with_codes.jsonl")
CLEAN_JSONL       = '/content/train_clean.jsonl'  # Geçici RAM'de

MIN_TEXT_LEN  = 10
MAX_TEXT_LEN  = 400
MAX_CODEC_LEN = 600

def clean_text(text: str) -> str:
    text = text.replace("'", "").replace("\u2019", "")
    text = text.replace('"', '').replace("\u201c", "").replace("\u201d", "")
    text = re.sub(r'\s+', ' ', text).strip()
    return text

with open(TRAIN_CODES_JSONL) as f:
    raw_lines = [l for l in f.readlines() if l.strip()]

clean_data = []
stats = {'cok_kisa': 0, 'cok_uzun': 0, 'uzun_codec': 0, 'temizlendi': 0, 'ok': 0}

for line in raw_lines:
    item  = json.loads(line)
    text  = clean_text(item.get('text', ''))
    codes = item.get('audio_codes', [])

    if len(text) < MIN_TEXT_LEN:  stats['cok_kisa'] += 1;   continue
    if len(text) > MAX_TEXT_LEN:  stats['cok_uzun'] += 1;   continue
    if len(codes) > MAX_CODEC_LEN: stats['uzun_codec'] += 1; continue
    if text != item.get('text', ''): stats['temizlendi'] += 1

    item['text'] = text

    audio_path = item.get('audio', '')
    if audio_path and os.path.isfile(audio_path):
        item['ref_audio'] = audio_path
    else:
        stats['cok_kisa'] += 1; continue

    clean_data.append(item)
    stats['ok'] += 1

print(f"Orijinal             : {len(raw_lines):,}")
print(f"Çok kısa / eksik     : {stats['cok_kisa']:,}")
print(f"Çok uzun (>{MAX_TEXT_LEN} kar)  : {stats['cok_uzun']:,}")
print(f"Uzun codec (>{MAX_CODEC_LEN})   : {stats['uzun_codec']:,}")
print(f"Metin düzeltilen     : {stats['temizlendi']:,}")
print(f"✓ Eğitime girecek    : {stats['ok']:,}")

with open(CLEAN_JSONL, 'w', encoding='utf-8') as f:
    for item in clean_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f"\nTemiz JSONL : {CLEAN_JSONL}")

---
## 7️⃣ Referans Ses 24kHz Dönüşümü

Inference için kullanılacak referans sesi Drive'a 24kHz olarak kaydeder.
Oturum kapansa da kaybolmaz.

In [ ]:
import torchaudio, json, os

REF_24K_PATH = os.path.join(OUTPUT_BASE, 'ceren_24k.wav')

if not os.path.exists(REF_24K_PATH):
    wav, sr = torchaudio.load(REF_AUDIO_PATH)
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != 24000:
        wav = torchaudio.functional.resample(wav, sr, 24000)
        print(f"Dönüştürüldü: {sr} Hz → 24000 Hz")
    torchaudio.save(REF_24K_PATH, wav, 24000)
    print(f"Drive'a kaydedildi: {REF_24K_PATH}")
else:
    print(f"Zaten mevcut: {REF_24K_PATH}")

ref_info = {'ref_audio': REF_24K_PATH, 'ref_text': REF_TEXT}
with open(os.path.join(OUTPUT_BASE, 'ref_info.json'), 'w') as f:
    json.dump(ref_info, f, ensure_ascii=False, indent=2)

print(f"\n✓ ref_info.json Drive'a kaydedildi")
print(f"  Eğitim    : her ses kendi referansı")
print(f"  Inference : {REF_24K_PATH}")

---
## 8️⃣ Eğitim Ayarları

Tüm parametreleri buradan değiştirin.

In [ ]:
import os

# ── Model ──────────────────────────────────────────────
INIT_MODEL   = 'Qwen/Qwen3-TTS-12Hz-1.7B-Base'
SPEAKER_NAME = 'turkce_ses'
CLEAN_START  = True   # True: eski checkpoint'leri sil, sıfırdan başla

# ── Temel parametreler ─────────────────────────────────
BATCH_SIZE    = 2      # A100 için güvenli
GRAD_ACCUM    = 16     # efektif batch = 32
LR            = 1e-5   # düşük LR = kararlı eğitim (5e-5 gürültüye sebep olur)
NUM_EPOCHS    = 15     # early stopping devreye girebilir
WARMUP_STEPS  = 500    # uzun warmup = kararlı başlangıç
MAX_CODEC_LEN = 600

# ── Kalite parametreleri ───────────────────────────────
VAL_RATIO         = 0.05
SEED              = 42
SUB_TALKER_WEIGHT = 0.3
NUM_DL_WORKERS    = 4
ATTN_IMPL         = 'sdpa'  # flash_attention_2 kuruluysa değiştir

# ── Yollar ────────────────────────────────────────────
LOGGING_DIR = os.path.join(OUTPUT_BASE, 'logs')
CLEAN_JSONL = '/content/train_clean.jsonl'
os.makedirs(LOGGING_DIR, exist_ok=True)

assert os.path.isfile(CLEAN_JSONL), f"Temiz JSONL yok: {CLEAN_JSONL} — Önce Hücre 6'yı çalıştırın."

toplam_saat = NUM_EPOCHS * 0.27 + 0.2

print("=" * 55)
print(" EĞİTİM AYARLARI — v16")
print("=" * 55)
print(f"  Model       : {INIT_MODEL}")
print(f"  LR          : {LR}")
print(f"  Batch       : {BATCH_SIZE} x {GRAD_ACCUM} = efektif {BATCH_SIZE*GRAD_ACCUM}")
print(f"  Epoch       : {NUM_EPOCHS}")
print(f"  Warmup      : {WARMUP_STEPS} step")
print(f"  ValRatio    : {VAL_RATIO}")
print(f"  AttnImpl    : {ATTN_IMPL}")
print(f"  Çıktı       : {OUTPUT_BASE}")
print("=" * 55)
print(f"  Tahmini süre  : ~{toplam_saat:.1f} saat")
print(f"  Tahmini kredi : ~{toplam_saat*17:.0f} / 95")
print("=" * 55)

---
## 9️⃣ Script Yazma

Bu hücreleri değiştirmeden çalıştırın.

In [ ]:
%%writefile /content/Qwen3-TTS/finetuning/dataset.py
# coding=utf-8
from typing import Tuple
import librosa
import numpy as np
import torch
from qwen_tts.core.models.configuration_qwen3_tts import Qwen3TTSConfig
from qwen_tts.core.models.modeling_qwen3_tts import mel_spectrogram
from torch.utils.data import Dataset

MAX_REF_MEL_FRAMES = 800

class TTSDataset(Dataset):
    def __init__(self, data_list, processor, config: Qwen3TTSConfig, lag_num=-1):
        self.data_list = data_list
        self.processor = processor
        self.lag_num   = lag_num
        self.config    = config
        tc = getattr(config, "talker_config", None)
        if isinstance(tc, dict):
            self.config.talker_config = type("TalkerConfig", (), tc)()

    def __len__(self):
        return len(self.data_list)

    def _load_audio_to_np(self, x: str) -> Tuple[np.ndarray, int]:
        audio, sr = librosa.load(x, sr=24000, mono=True)
        if audio.ndim > 1:
            audio = np.mean(audio, axis=-1)
        return audio.astype(np.float32), 24000

    def _normalize_audio_inputs(self, audios):
        out = []
        for a in (audios if isinstance(audios, list) else [audios]):
            if isinstance(a, str):
                out.append(self._load_audio_to_np(a))
            elif isinstance(a, tuple) and isinstance(a[0], np.ndarray):
                out.append((a[0].astype(np.float32), int(a[1])))
            else:
                raise TypeError(f"Desteklenmeyen ses tipi: {type(a)}")
        return out

    def _build_assistant_text(self, text):
        return f"<|im_start|>assistant\n{text}<|im_end|>\n<|im_start|>assistant\n"

    def _tokenize_texts(self, text):
        inp = self.processor(text=text, return_tensors="pt", padding=True)
        iid = inp["input_ids"]
        return iid.unsqueeze(0) if iid.dim() == 1 else iid

    @torch.inference_mode()
    def extract_mels(self, audio, sr):
        mel = mel_spectrogram(
            torch.from_numpy(audio).unsqueeze(0),
            n_fft=1024, num_mels=128, sampling_rate=24000,
            hop_size=256, win_size=1024, fmin=0, fmax=12000,
        ).transpose(1, 2)
        T = mel.shape[1]
        if T >= MAX_REF_MEL_FRAMES:
            start = (T - MAX_REF_MEL_FRAMES) // 2
            mel = mel[:, start:start + MAX_REF_MEL_FRAMES, :]
        else:
            mel = torch.nn.functional.pad(mel, (0, 0, 0, MAX_REF_MEL_FRAMES - T))
        return mel

    def __getitem__(self, idx):
        item        = self.data_list[idx]
        text        = self._build_assistant_text(item["text"])
        text_ids    = self._tokenize_texts(text)
        codes       = item.get("audio_codes") or item.get("codec_ids")
        audio_codes = torch.tensor(codes, dtype=torch.long)
        ref_list    = item["ref_audio"] if isinstance(item["ref_audio"], list) else [item["ref_audio"]]
        wav, sr     = self._normalize_audio_inputs(ref_list)[0]
        ref_mel     = self.extract_mels(audio=wav, sr=sr)
        return {"text_ids": text_ids[:, :-5], "audio_codes": audio_codes, "ref_mel": ref_mel}

    def collate_fn(self, batch):
        item_length = [b["text_ids"].shape[1] + b["audio_codes"].shape[0] for b in batch]
        max_length  = max(item_length) + 8
        b, t = len(batch), max_length
        input_ids            = torch.zeros((b, t, 2),  dtype=torch.long)
        codec_ids            = torch.zeros((b, t, 16), dtype=torch.long)
        text_embedding_mask  = torch.zeros((b, t), dtype=torch.bool)
        codec_embedding_mask = torch.zeros((b, t), dtype=torch.bool)
        codec_mask           = torch.zeros((b, t), dtype=torch.bool)
        attention_mask       = torch.zeros((b, t), dtype=torch.long)
        codec_0_labels       = torch.full((b, t), -100, dtype=torch.long)
        tc = self.config.talker_config
        for i, data in enumerate(batch):
            tid = data["text_ids"]
            ac0 = data["audio_codes"][:, 0]
            acs = data["audio_codes"]
            tl  = tid.shape[1]
            cl  = ac0.shape[0]
            input_ids[i, :3, 0]                   = tid[0, :3]
            input_ids[i, 3:7, 0]                  = self.config.tts_pad_token_id
            input_ids[i, 7, 0]                    = self.config.tts_bos_token_id
            input_ids[i, 8:8+tl-3, 0]             = tid[0, 3:]
            input_ids[i, 8+tl-3, 0]               = self.config.tts_eos_token_id
            input_ids[i, 8+tl-2:8+tl+cl, 0]      = self.config.tts_pad_token_id
            text_embedding_mask[i, :8+tl+cl]      = True
            input_ids[i, 3:8, 1]                  = torch.tensor([
                tc.codec_nothink_id, tc.codec_think_bos_id,
                tc.codec_think_eos_id, 0, tc.codec_pad_id])
            input_ids[i, 8:8+tl-3, 1]             = tc.codec_pad_id
            input_ids[i, 8+tl-3, 1]               = tc.codec_pad_id
            input_ids[i, 8+tl-2, 1]               = tc.codec_bos_id
            input_ids[i, 8+tl-1:8+tl-1+cl, 1]    = ac0
            input_ids[i, 8+tl-1+cl, 1]            = tc.codec_eos_token_id
            codec_0_labels[i, 8+tl-1:8+tl-1+cl]  = ac0
            codec_0_labels[i, 8+tl-1+cl]          = tc.codec_eos_token_id
            codec_ids[i, 8+tl-1:8+tl-1+cl, :]    = acs
            codec_embedding_mask[i, 3:8+tl+cl]    = True
            codec_embedding_mask[i, 6]             = False
            codec_mask[i, 8+tl-1:8+tl-1+cl]       = True
            attention_mask[i, :8+tl+cl]            = True
        ref_mels = torch.cat([d["ref_mel"] for d in batch], dim=0)
        return {
            "input_ids": input_ids, "ref_mels": ref_mels,
            "attention_mask": attention_mask,
            "text_embedding_mask": text_embedding_mask.unsqueeze(-1),
            "codec_embedding_mask": codec_embedding_mask.unsqueeze(-1),
            "codec_0_labels": codec_0_labels,
            "codec_ids": codec_ids,
            "codec_mask": codec_mask,
        }


In [ ]:
%%writefile /content/Qwen3-TTS/finetuning/sft_v16.py
# coding=utf-8
# sft_v16.py — Drive çıktılı, val_loss + speaker_sim izleme, early stopping, resume
# Düzeltmeler: hidden_states[-1], spk_emb her batch güncellenir, project_dir doğru
import argparse, json, os, re, shutil, sys

for p in ['/content/Qwen3-TTS', '/content/Qwen3-TTS/finetuning']:
    if p not in sys.path:
        sys.path.insert(0, p)

import torch
import torch.nn.functional as F
from accelerate import Accelerator
from dataset import TTSDataset
from huggingface_hub import snapshot_download
from qwen_tts.inference.qwen3_tts_model import Qwen3TTSModel
from safetensors.torch import save_file, load_file
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoConfig, get_cosine_schedule_with_warmup

target_speaker_embedding = None

def _read_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

def _codec_len(d):
    c = d.get('audio_codes') or d.get('codec_ids') or []
    return len(c)

def _split(items, val_ratio, seed):
    if val_ratio <= 0: return items, []
    g = torch.Generator(); g.manual_seed(seed)
    perm = torch.randperm(len(items), generator=g).tolist()
    n_val = max(1, int(round(len(items) * val_ratio)))
    val_idx = set(perm[:n_val])
    train, val = [], []
    for i, it in enumerate(items):
        (val if i in val_idx else train).append(it)
    return train, val

def find_best_checkpoint(output_path):
    best_vl, best_ep, best_path = float('inf'), -1, None
    for name in os.listdir(output_path):
        m = re.search(r'checkpoint-epoch-(\d+)$', name)
        if not m: continue
        ep = int(m.group(1))
        meta_file  = os.path.join(output_path, name, 'meta.json')
        model_file = os.path.join(output_path, name, 'model.safetensors')
        if not os.path.isfile(model_file): continue
        if os.path.isfile(meta_file):
            meta = json.load(open(meta_file))
            vl = meta.get('val_loss')
            if vl is not None and vl < best_vl:
                best_vl, best_ep, best_path = vl, ep, os.path.join(output_path, name)
    return best_path, best_ep, best_vl

def save_checkpoint(accelerator, model, MODEL_PATH, out_dir, speaker_name, spk_emb):
    os.makedirs(out_dir, exist_ok=True)
    for fname in os.listdir(MODEL_PATH):
        if fname == 'model.safetensors': continue
        src = os.path.join(MODEL_PATH, fname)
        dst = os.path.join(out_dir, fname)
        if not os.path.exists(dst):
            if os.path.isfile(src): shutil.copy2(src, dst)
            elif os.path.isdir(src): shutil.copytree(src, dst, dirs_exist_ok=True)
    with open(os.path.join(MODEL_PATH, 'config.json')) as f:
        cfg = json.load(f)
    cfg['tts_model_type'] = 'custom_voice'
    tc = cfg.get('talker_config', {})
    tc['spk_id']         = {speaker_name: 3000}
    tc['spk_is_dialect'] = {speaker_name: False}
    cfg['talker_config'] = tc
    with open(os.path.join(out_dir, 'config.json'), 'w') as f:
        json.dump(cfg, f, indent=2, ensure_ascii=False)
    unwrapped  = accelerator.unwrap_model(model)
    state_dict = {k: v.detach().cpu() for k, v in unwrapped.state_dict().items()}
    w = state_dict['talker.model.codec_embedding.weight']
    state_dict['talker.model.codec_embedding.weight'][3000] = (
        spk_emb[0].detach().to(w.device).to(w.dtype)
    )
    save_file(state_dict, os.path.join(out_dir, 'model.safetensors'))

def train():
    global target_speaker_embedding
    p = argparse.ArgumentParser()
    p.add_argument('--init_model_path',   default='Qwen/Qwen3-TTS-12Hz-1.7B-Base')
    p.add_argument('--output_path',       required=True)
    p.add_argument('--train_jsonl',       required=True)
    p.add_argument('--val_ratio',         type=float, default=0.05)
    p.add_argument('--seed',              type=int,   default=42)
    p.add_argument('--batch_size',        type=int,   default=2)
    p.add_argument('--lr',                type=float, default=1e-5)
    p.add_argument('--num_epochs',        type=int,   default=15)
    p.add_argument('--grad_accum',        type=int,   default=16)
    p.add_argument('--warmup_steps',      type=int,   default=500)
    p.add_argument('--max_codec_len',     type=int,   default=600)
    p.add_argument('--speaker_name',      default='turkce_ses')
    p.add_argument('--logging_dir',       default='./logs')
    p.add_argument('--sub_talker_weight', type=float, default=0.3)
    p.add_argument('--num_workers',       type=int,   default=4)
    p.add_argument('--attn_impl',         default='sdpa')
    p.add_argument('--resume',            action='store_true')
    p.add_argument('--start_epoch',       type=int,   default=0)
    args = p.parse_args()

    os.makedirs(args.output_path, exist_ok=True)
    os.makedirs(args.logging_dir, exist_ok=True)

    accelerator = Accelerator(
        gradient_accumulation_steps=args.grad_accum,
        mixed_precision='bf16',
        log_with='tensorboard',
        project_dir=args.logging_dir,
    )

    MODEL_PATH = snapshot_download(args.init_model_path)
    accelerator.print(f'Model: {MODEL_PATH}')

    qwen3tts = Qwen3TTSModel.from_pretrained(
        MODEL_PATH, dtype=torch.bfloat16,
        attn_implementation=args.attn_impl,
    )
    config = AutoConfig.from_pretrained(MODEL_PATH)

    all_items = _read_jsonl(args.train_jsonl)
    before    = len(all_items)
    all_items = [d for d in all_items if _codec_len(d) <= args.max_codec_len]
    accelerator.print(f'Veri: {before:,} → {len(all_items):,}')

    train_items, val_items = _split(all_items, args.val_ratio, args.seed)
    accelerator.print(f'Train: {len(train_items):,} | Val: {len(val_items):,}')

    ds = TTSDataset(train_items, qwen3tts.processor, config)
    loader = DataLoader(ds, batch_size=args.batch_size, shuffle=True,
                        collate_fn=ds.collate_fn, num_workers=args.num_workers,
                        pin_memory=True, persistent_workers=True, prefetch_factor=4)

    val_loader = None
    if val_items:
        vds = TTSDataset(val_items, qwen3tts.processor, config)
        val_loader = DataLoader(vds, batch_size=args.batch_size, shuffle=False,
                                collate_fn=vds.collate_fn, num_workers=2,
                                pin_memory=True, persistent_workers=True, prefetch_factor=2)

    optimizer     = AdamW(qwen3tts.model.parameters(), lr=args.lr, weight_decay=0.01)
    opt_per_epoch = len(loader) // args.grad_accum
    total_steps   = opt_per_epoch * args.num_epochs
    accelerator.print(f'Steps/epoch: {opt_per_epoch} | Toplam: {total_steps}')

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=args.warmup_steps,
        num_training_steps=total_steps,
    )

    model, optimizer, loader, scheduler = accelerator.prepare(
        qwen3tts.model, optimizer, loader, scheduler)
    if val_loader:
        val_loader = accelerator.prepare(val_loader)

    start_epoch      = args.start_epoch
    best_val_loss    = float('inf')
    best_val_spk_sim = float('-inf')
    best_epoch       = -1
    no_improve       = 0
    PATIENCE         = 5

    if args.resume:
        resume_ckpt, resume_ep, resume_vl = find_best_checkpoint(args.output_path)
        if resume_ckpt:
            weights_path = os.path.join(resume_ckpt, 'model.safetensors')
            accelerator.print(f'Resume: val_loss={resume_vl:.4f}, E{resume_ep:02d}')
            ckpt_weights = load_file(weights_path, device=str(accelerator.device))
            unwrapped = accelerator.unwrap_model(model)
            missing, unexpected = unwrapped.load_state_dict(ckpt_weights, strict=False)
            accelerator.print(f'Yüklendi — missing={len(missing)} unexpected={len(unexpected)}')
            start_epoch   = resume_ep + 1
            best_val_loss = resume_vl
            best_epoch    = resume_ep
        else:
            accelerator.print('Resume checkpoint bulunamadı, sıfırdan başlıyor.')

    def compute_loss(batch):
        inp   = batch['input_ids'].to(model.device)
        cids  = batch['codec_ids'].to(model.device)
        rmels = batch['ref_mels'].to(model.device)
        temb  = batch['text_embedding_mask'].to(model.device)
        cemb  = batch['codec_embedding_mask'].to(model.device)
        amask = batch['attention_mask'].to(model.device)
        c0lab = batch['codec_0_labels'].to(model.device)
        cmask = batch['codec_mask'].to(model.device)

        spk_emb = model.speaker_encoder(rmels.to(model.dtype))
        t_ids   = inp[:, :, 0]; c_ids = inp[:, :, 1]
        temb_v  = model.talker.model.text_embedding(t_ids)  * temb
        cemb_v  = model.talker.model.codec_embedding(c_ids) * cemb
        cemb_v[:, 6, :] = spk_emb
        x = temb_v + cemb_v
        for i in range(1, 16):
            cb = model.talker.code_predictor.get_input_embeddings()[i-1](cids[:, :, i])
            x  = x + cb * cmask.unsqueeze(-1)

        out = model.talker(
            inputs_embeds=x[:, :-1, :].to(model.dtype),
            attention_mask=amask[:, :-1],
            labels=c0lab[:, 1:],
            output_hidden_states=True,
        )
        hs    = out.hidden_states[-1]          # v15 düzeltmesi
        ths   = hs[cmask[:, 1:]]
        tcids = cids[cmask]
        _, sub_loss = model.talker.forward_sub_talker_finetune(tcids, ths)
        loss = out.loss + args.sub_talker_weight * sub_loss
        return loss, out.loss, sub_loss, spk_emb

    for epoch in range(start_epoch, args.num_epochs):
        model.train()
        ep_loss, ep_cnt = 0.0, 0

        for step, batch in enumerate(loader):
            with accelerator.accumulate(model):
                loss, ml, sl, spk = compute_loss(batch)
                target_speaker_embedding = spk.detach()  # v15 düzeltmesi
                accelerator.backward(loss)
                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step(); scheduler.step(); optimizer.zero_grad()
            ep_loss += loss.item(); ep_cnt += 1
            if step % 20 == 0:
                accelerator.print(
                    f'E{epoch:02d} S{step:04d} | loss={loss.item():.4f} '
                    f'(m={ml.item():.4f} s={sl.item():.4f}) | '
                    f'lr={scheduler.get_last_lr()[0]:.2e}')

        avg_loss = ep_loss / max(ep_cnt, 1)
        val_loss, val_spk_sim = None, None

        if val_loader:
            model.eval()
            v_sum, v_cnt, s_sum, s_cnt = 0., 0, 0., 0
            tv = model.talker.model.codec_embedding.weight[3000].detach()
            with torch.no_grad():
                for vb in val_loader:
                    vl, _, _, vspk = compute_loss(vb)
                    v_sum += float(vl); v_cnt += 1
                    sims = F.cosine_similarity(
                        vspk.detach(),
                        tv.unsqueeze(0).expand(vspk.shape[0], -1).to(vspk.device, vspk.dtype),
                        dim=-1)
                    s_sum += float(sims.mean()); s_cnt += 1
            val_loss    = v_sum / max(v_cnt, 1)
            val_spk_sim = s_sum / max(s_cnt, 1)

        accelerator.print(
            f'=== Epoch {epoch:02d} | avg={avg_loss:.4f} | '
            f'val={val_loss:.4f} | spk_sim={val_spk_sim:.4f} ===')

        if accelerator.is_main_process:
            ckpt = os.path.join(args.output_path, f'checkpoint-epoch-{epoch:02d}')
            save_checkpoint(accelerator, model, MODEL_PATH, ckpt,
                            args.speaker_name, target_speaker_embedding)
            meta = {'epoch': epoch, 'avg_loss': avg_loss,
                    'val_loss': val_loss, 'val_spk_sim': val_spk_sim}
            json.dump(meta, open(os.path.join(ckpt, 'meta.json'), 'w'), indent=2)
            accelerator.print(f'✓ Checkpoint Drive'a kaydedildi: {ckpt}')

            is_best = (val_spk_sim is not None and
                       (val_spk_sim > best_val_spk_sim or
                        (val_spk_sim == best_val_spk_sim and (val_loss or avg_loss) < best_val_loss)))
            if not is_best and val_spk_sim is None:
                is_best = avg_loss < best_val_loss

            if is_best:
                best_val_loss    = val_loss or avg_loss
                best_val_spk_sim = val_spk_sim or 0.
                best_epoch = epoch; no_improve = 0
                best_dir = os.path.join(args.output_path, 'checkpoint-best')
                if os.path.isdir(best_dir): shutil.rmtree(best_dir)
                shutil.copytree(ckpt, best_dir)
                accelerator.print(f'*** Yeni best: E{epoch:02d} spk_sim={best_val_spk_sim:.4f} ***')
            else:
                no_improve += 1
                accelerator.print(f'İyileşme yok: {no_improve}/{PATIENCE}')
                if no_improve >= PATIENCE:
                    accelerator.print(f'Early stopping! Best epoch={best_epoch}')
                    break

    accelerator.print(f'Eğitim bitti. Best epoch={best_epoch} val_loss={best_val_loss:.4f}')

if __name__ == '__main__':
    train()


---
## 🔟 Eğitim 🚀

Her checkpoint direkt **Drive'a** kaydedilir.
Early stopping: 5 epoch iyileşme olmadığında durur.
Resume: Oturum kapanırsa `--resume` ekleyerek kaldığı yerden devam edilebilir.

In [ ]:
import os, shutil, json

if CLEAN_START:
    removed = 0
    for name in os.listdir(OUTPUT_BASE):
        if name.startswith('checkpoint-'):
            p = os.path.join(OUTPUT_BASE, name)
            if os.path.isdir(p):
                shutil.rmtree(p)
                removed += 1
    print(f"Temizlendi: {removed} eski checkpoint silindi" if removed else "Temizlenecek checkpoint yok")
else:
    print("CLEAN_START=False — mevcut checkpoint'ler korunuyor")

%cd /content/Qwen3-TTS/finetuning

!PYTHONPATH=/content/Qwen3-TTS python sft_v16.py \
  --init_model_path   {INIT_MODEL} \
  --output_path       {OUTPUT_BASE} \
  --train_jsonl       {CLEAN_JSONL} \
  --speaker_name      {SPEAKER_NAME} \
  --batch_size        {BATCH_SIZE} \
  --lr                {LR} \
  --num_epochs        {NUM_EPOCHS} \
  --grad_accum        {GRAD_ACCUM} \
  --warmup_steps      {WARMUP_STEPS} \
  --max_codec_len     {MAX_CODEC_LEN} \
  --logging_dir       {LOGGING_DIR} \
  --val_ratio         {VAL_RATIO} \
  --seed              {SEED} \
  --sub_talker_weight {SUB_TALKER_WEIGHT} \
  --num_workers       {NUM_DL_WORKERS} \
  --attn_impl         {ATTN_IMPL}

# Drive'daki checkpoint özeti
print("\n📁 Drive'daki checkpoint'ler:")
for name in sorted(os.listdir(OUTPUT_BASE)):
    if name.startswith('checkpoint-'):
        meta_path = os.path.join(OUTPUT_BASE, name, 'meta.json')
        if os.path.isfile(meta_path):
            meta = json.load(open(meta_path))
            print(f"  📌 {name} — val_loss={meta.get('val_loss','?'):.4f}  spk_sim={meta.get('val_spk_sim','?'):.4f}")
        else:
            print(f"  📌 {name}")

---
## 1️⃣1️⃣ Inference Testi 🎤

Eğitim bittikten sonra çalıştırın. Test sesleri Drive'a kaydedilir.

In [ ]:
import json, os, sys, torch
import soundfile as sf
from IPython.display import Audio, display

sys.path.insert(0, '/content/Qwen3-TTS')
from qwen_tts.inference.qwen3_tts_model import Qwen3TTSModel

best_dir = os.path.join(OUTPUT_BASE, 'checkpoint-best')
if not os.path.isdir(best_dir):
    existing = sorted([d for d in os.listdir(OUTPUT_BASE) if d.startswith('checkpoint-epoch-')])
    assert existing, "Hiç checkpoint bulunamadı! Önce eğitimi çalıştırın."
    best_dir = os.path.join(OUTPUT_BASE, existing[-1])

print(f"Checkpoint: {best_dir}")
tts = Qwen3TTSModel.from_pretrained(best_dir)

test_sentences = [
    "Merhaba, bugün hava çok güzel.",
    "Türkiye'nin başkenti Ankara'dır.",
    "Yapay zeka teknolojileri son yıllarda inanılmaz hızda gelişmektedir.",
    "Güzel şehrimizde yağmur yağıyor, şemsiyeni almayı unutma.",
    "Bu cümle Türkçe'nin ğ, ş, ç, ı, ö ve ü harflerini içermektedir.",
]

for i, text in enumerate(test_sentences):
    print(f"\n--- Test {i+1} ---")
    print(f"Metin: {text}")
    try:
        wavs, sr = tts.generate_custom_voice(
            text=text,
            speaker=SPEAKER_NAME,
            language="auto",
            non_streaming_mode=True,
        )
        out_path = os.path.join(OUTPUT_BASE, f'test_{i+1}.wav')
        sf.write(out_path, wavs[0], sr)
        print(f"Drive'a kaydedildi: {out_path}")
        display(Audio(wavs[0], rate=sr))
    except Exception as e:
        print(f"HATA: {e}")

---
## 1️⃣2️⃣ Bellek Temizliği

In [ ]:
import torch, gc
try: del tts
except: pass
gc.collect()
torch.cuda.empty_cache()
if torch.cuda.is_available():
    print(f'Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB')
    print(f'Reserved : {torch.cuda.memory_reserved()/1e9:.2f} GB')
print("✓ Temizlik tamamlandı.")